# CleanOps — YOLOv8n Baseline Waste Detection Training

**Objective:** train a baseline **YOLOv8n** (nano) detector on the **cleaned TACO** dataset, using the class mapping defined in **Task 3 (Waste Dataset Audit, Category Analysis & Explanation Report)**.

**Deliverables produced by this notebook**

| File | What it is |
|---|---|
| `ai/training/cleanops_yolov8_training.ipynb` | this notebook (download it *after* running, so the outputs are saved in it) |
| `ai/models/yolov8n_cleanops_best.pt` | best weights found during training |
| `ai/models/metrics.json` | validation metrics: mAP50, mAP50-95, precision, recall |

**Before you run (3 things)**
1. Colab: *Runtime → Change runtime type → T4 GPU*.
2. Put `TACO.zip` (the Kaggle *TACO Dataset YOLO Format* file) and `clean_taco.py` in a Google Drive folder named `cleanops` (`My Drive/cleanops/`).
3. Paste the Task-3 class mapping into the `CLASS_MAP` cell (section 4). Leave it as `None` only if Task 3 keeps the 18 original TACO classes.

**Pipeline:** raw TACO zip → `clean_taco.py` (audit, clean, leakage-free split) → class mapping from Task 3 → YOLOv8n training (50 epochs, `imgsz=640`, `batch=16`) → validation metrics → inference test.

> This is the **baseline** run with the nano model, as specified by the task. Larger variants (s / m) are compared later, as described in `docs/design/ai/yolov8-detection-spec.md`.

## 1. Environment

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected. Runtime -> Change runtime type -> T4 GPU, then run again.")

In [ ]:
!pip -q install ultralytics

## 2. Google Drive and paths

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR    = '/content/drive/MyDrive/cleanops'      # folder that holds TACO.zip and clean_taco.py
TACO_ZIP     = f'{DRIVE_DIR}/TACO.zip'
CLEAN_SCRIPT = f'{DRIVE_DIR}/clean_taco.py'

for p in (TACO_ZIP, CLEAN_SCRIPT):
    print(('OK       ' if os.path.exists(p) else 'MISSING  ') + p)

## 3. Clean the TACO dataset

`clean_taco.py` audits every label and image, removes invalid boxes, groups flipped/rotated copies of the same photo, and writes a **leakage-free, class-stratified 80/10/10 split** to `/content/TACO_clean`
(see `docs/design/ai/taco-dataset-analysis.md` for why the original TACO split cannot be used).

In [ ]:
import zipfile
from pathlib import Path

RAW = Path('/content/TACO_raw')
if not RAW.exists():
    with zipfile.ZipFile(TACO_ZIP) as z:          # TACO.zip = the real dataset (about 243 MB)
        z.extractall(RAW)
TACO_SRC = next(RAW.rglob('data.yaml')).parent      # folder that contains data.yaml, train/, valid/, test/
print('TACO folder:', TACO_SRC)

In [ ]:
!python "{CLEAN_SCRIPT}" --src "{TACO_SRC}" --out /content/TACO_clean --report /content/TACO_clean_report.json

## 4. Apply the class mapping from Task 3

* `CLASS_MAP = None` keeps the 18 original TACO classes.
* Otherwise map **every** TACO class name to a Task-3 category (several TACO classes can share one category). Use `None` as the target to drop a class — images left without any box are removed.
* The cell refuses to run twice on the same data (re-run section 3 first), so labels can never be remapped twice by mistake.

In [ ]:
from pathlib import Path
import yaml

CLEAN = Path('/content/TACO_clean')
taco_names = yaml.safe_load((CLEAN / 'data.yaml').read_text())['names']
print(len(taco_names), 'TACO classes:', taco_names)

# ---- PASTE THE CLASS MAPPING FROM TASK 3 HERE ---------------------------------------------
# Format: {'<TACO class name>': '<Task-3 category>', ...}     (target None = drop that class)
# Example of the format only (replace with the real Task-3 mapping):
#   CLASS_MAP = {'Bottle': 'Plastic', 'Bottle cap': 'Plastic', 'Broken glass': 'Glass', ...}
CLASS_MAP = None        # None = keep the 18 original TACO classes
# -------------------------------------------------------------------------------------------

marker = CLEAN / '.class_map_applied'
if marker.exists():
    raise RuntimeError('Mapping was already applied to these labels. Re-run section 3 (cleaning) first, then this cell.')

if CLASS_MAP is None:
    new_names = list(taco_names)
    old2new = {i: i for i in range(len(taco_names))}
else:
    missing = [n for n in taco_names if n not in CLASS_MAP]
    unknown = [k for k in CLASS_MAP if k not in taco_names]
    assert not missing, f'Add these TACO classes to CLASS_MAP (use None to drop): {missing}'
    assert not unknown, f'These keys are not TACO class names (check spelling): {unknown}'
    new_names = []
    for n in taco_names:
        target = CLASS_MAP[n]
        if target is not None and target not in new_names:
            new_names.append(target)
    old2new = {i: (None if CLASS_MAP[n] is None else new_names.index(CLASS_MAP[n]))
               for i, n in enumerate(taco_names)}

removed_images = 0
if CLASS_MAP is not None:
    for split in ('train', 'valid', 'test'):
        for lbl in sorted((CLEAN / split / 'labels').glob('*.txt')):
            out, seen = [], set()
            for line in lbl.read_text().splitlines():
                p = line.split()
                if len(p) != 5:
                    continue
                new_id = old2new[int(p[0])]
                if new_id is None:
                    continue
                key = (new_id, *p[1:])
                if key in seen:                      # identical box produced by merging two classes
                    continue
                seen.add(key)
                out.append(' '.join([str(new_id), *p[1:]]))
            if out:
                lbl.write_text('\n'.join(out) + '\n')
            else:
                lbl.unlink()
                for img in (CLEAN / split / 'images').glob(lbl.stem + '.*'):
                    img.unlink()
                removed_images += 1
marker.write_text('applied')

data_cfg = {'path': str(CLEAN), 'train': 'train/images', 'val': 'valid/images', 'test': 'test/images',
            'names': {i: n for i, n in enumerate(new_names)}}
Path('/content/cleanops_data.yaml').write_text(yaml.safe_dump(data_cfg, sort_keys=False, allow_unicode=True))

print(f'Classes used for training ({len(new_names)}):', new_names)
print('Images removed because all their boxes were dropped:', removed_images)
print(Path('/content/cleanops_data.yaml').read_text())

### Sanity check — images and boxes per split and class

In [ ]:
from collections import Counter

for split in ('train', 'valid', 'test'):
    n_img = len(list((CLEAN / split / 'images').glob('*')))
    cnt = Counter()
    for lbl in (CLEAN / split / 'labels').glob('*.txt'):
        for line in lbl.read_text().splitlines():
            if line.strip():
                cnt[int(line.split()[0])] += 1
    print(f'{split:5s}: {n_img} images, {sum(cnt.values())} boxes')
    if split == 'train':
        for i, n in enumerate(new_names):
            print(f'        {n:24s} {cnt[i]}')

## 5. Train YOLOv8n

Exactly the configuration given in the task: YOLOv8 **nano**, COCO-pretrained (`yolov8n.pt`), `epochs=50`, `imgsz=640`, `batch=16`.

In [ ]:
import os, json
from ultralytics import YOLO

os.chdir('/content')                     # 'cleanops_data.yaml' is resolved from here

# 1. Initialize YOLOv8 nano model
model = YOLO('yolov8n.pt')

# 2. Train on cleaned TACO dataset
results = model.train(
    data='cleanops_data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    name='cleanops_yolov8n'
)

## 6. Export the best weights and the validation metrics

In [ ]:
import shutil
from pathlib import Path

os.makedirs('ai/models', exist_ok=True)

# best weights found during training -> deliverable path
save_dir = Path(model.trainer.save_dir)
shutil.copy(save_dir / 'weights' / 'best.pt', 'ai/models/yolov8n_cleanops_best.pt')

# 3. Export validation metrics (the model now holds the best weights; validation split = 'valid')
metrics = model.val()
summary = {
    "mAP50": float(metrics.box.map50),
    "mAP50-95": float(metrics.box.map),
    "precision": float(metrics.box.mp),
    "recall": float(metrics.box.mr)
}
with open('ai/models/metrics.json', 'w') as f:
    json.dump(summary, f, indent=4)

print(json.dumps(summary, indent=4))

### Optional — the same model on the held-out test split (printed only, not stored in `metrics.json`)

The test split is leakage-free (no photo appears in both train and test), so this number is the honest estimate to quote in the report.

In [ ]:
test_metrics = model.val(split='test')
print('TEST  mAP50    :', round(float(test_metrics.box.map50), 4))
print('TEST  mAP50-95 :', round(float(test_metrics.box.map), 4))
print('TEST  precision:', round(float(test_metrics.box.mp), 4))
print('TEST  recall   :', round(float(test_metrics.box.mr), 4))

## 7. Inference test

Runs the saved weights on one street photo. By default the first image of the test split is used; to use your own photo, upload it to Colab as `sample_street_waste.jpg` first.

In [ ]:
import glob
from IPython.display import Image, display

if not os.path.exists('sample_street_waste.jpg'):
    first_test_image = sorted(glob.glob('/content/TACO_clean/test/images/*'))[0]
    shutil.copy(first_test_image, 'sample_street_waste.jpg')

model = YOLO('ai/models/yolov8n_cleanops_best.pt')
results = model.predict(source='sample_street_waste.jpg', save=True)

out_dir = Path(results[0].save_dir)               # e.g. runs/detect/predict
saved = sorted(out_dir.glob('*.jpg'))[0]
print('Predictions saved in:', out_dir)
print('Detections:', [(model.names[int(c)], round(float(s), 2)) for c, s in zip(results[0].boxes.cls, results[0].boxes.conf)])
display(Image(filename=str(saved)))

In [ ]:
# Verify that metrics.json records the validation mAP50
print(open('ai/models/metrics.json').read())

## 8. Save the deliverables

Creates `cleanops_outputs.zip` (contains `ai/models/yolov8n_cleanops_best.pt` and `ai/models/metrics.json`), copies it to Drive and downloads it.
Then download **this notebook with its outputs** from Colab: *File → Download → Download .ipynb*.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('/content/cleanops_outputs', 'zip', root_dir='/content', base_dir='ai')
shutil.copy('/content/cleanops_outputs.zip', f'{DRIVE_DIR}/cleanops_outputs.zip')
files.download('/content/cleanops_outputs.zip')